# GxP-LLM results dashboard

Attach the saved outputs of the evaluation, fine-tuning, and quantization notebooks. This notebook scans their JSON results, creates a comparable table and CSV, and optionally publishes the table to W&B. The W&B dashboard remains the live view; this is a portable report artifact.

In [ ]:
%pip install -q pandas matplotlib seaborn wandb

from pathlib import Path
import json, os
import pandas as pd

# Kaggle attaches notebook outputs below /kaggle/input. The scan also works if
# you upload a combined results dataset instead.
INPUT_ROOT = Path('/kaggle/input')
OUTPUT_DIR = Path('/kaggle/working/gxp_results_dashboard')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
result_files = list(INPUT_ROOT.rglob('full_results.json'))
if not result_files:
    raise FileNotFoundError('Attach at least one evaluation or quantization notebook output.')
print('Found:', *result_files, sep='\n- ')

In [ ]:
def flatten_result(path):
    payload = json.loads(path.read_text(encoding='utf-8'))
    rows = []
    for split, result in payload.items():
        metrics = result.get('metrics', {})
        rouge = metrics.get('rougeL', {})
        judge = result.get('judge_scores', {})
        judge_values = [v for group in judge.values() for k, v in group.items() if isinstance(v, (int, float))]
        adversarial = result.get('adversarial', {})
        rows.append({
            'artifact': path.parent.name,
            'result_file': str(path),
            'split': split,
            'exact_match': metrics.get('exact_match'),
            'rougeL_f1': rouge.get('rougeL_f1'),
            'bleu': metrics.get('bleu'),
            'judge_mean': sum(judge_values) / len(judge_values) if judge_values else None,
            'refusal_rate': adversarial.get('refusal_rate'),
            'false_compliance_rate': adversarial.get('false_compliance_rate'),
            'helpful_redirect_rate': adversarial.get('helpful_redirect_rate'),
        })
    return rows

rows = [row for path in result_files for row in flatten_result(path)]
table = pd.DataFrame(rows).sort_values(['artifact', 'split']).reset_index(drop=True)
display(table)

In [ ]:
table.to_csv(OUTPUT_DIR / 'results_comparison.csv', index=False)
(OUTPUT_DIR / 'results_comparison.json').write_text(table.to_json(orient='records', indent=2), encoding='utf-8')

import matplotlib.pyplot as plt
plot = table.dropna(subset=['rougeL_f1']).pivot_table(index='artifact', columns='split', values='rougeL_f1')
if not plot.empty:
    ax = plot.plot(kind='bar', figsize=(12, 5), title='ROUGE-L F1 by model stage')
    ax.set_ylabel('ROUGE-L F1'); plt.tight_layout(); plt.savefig(OUTPUT_DIR / 'rougeL_comparison.png', dpi=160)
print(f'Portable dashboard outputs written to {OUTPUT_DIR}')

In [ ]:
# Optional W&B publication. Add WANDB_API_KEY as a Kaggle Secret to enable it.
try:
    from kaggle_secrets import UserSecretsClient
    import wandb
    key = UserSecretsClient().get_secret('WANDB_API_KEY')
    os.environ['WANDB_API_KEY'] = key
    with wandb.init(project='gxp-llm', name='results-dashboard', job_type='report', config={'n_rows': len(table)}) as run:
        run.log({'results_table': wandb.Table(dataframe=table)})
        artifact = wandb.Artifact('gxp-results-dashboard', type='report')
        artifact.add_dir(str(OUTPUT_DIR)); run.log_artifact(artifact)
    print('Published dashboard table and files to W&B.')
except Exception as exc:
    print(f'W&B publication skipped: {exc}')